# Entrainement du modele EcoSort — Transfer Learning (MobileNetV2)

Ce notebook entraine le modele de classification sur les 5 categories officielles
(`jaune`, `verte`, `bleue`, `d3e`, `marron`) a partir du dataset deja reorganise par
`prepare_dataset.py` (`data/processed/{train,val}`).

Strategie :
- **Phase 1** : backbone MobileNetV2 gele, entrainement de la tete de classification.
- **Phase 2** : fine-tuning des dernieres couches du backbone a tres faible learning rate.
- **Augmentation renforcee** (contraste, luminosite, zoom, translation) pour reduire
  l'ecart entre les photos "propres" de Kaggle et les photos e-commerce Jumia.

Une fois que les resultats ici sont satisfaisants, ce notebook sera converti en script
`src/ml/train.py` reproductible pour le livrable final.

In [ ]:
import json
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = Path("../data/processed")
OUTPUT_MODEL = Path("../models/modele_eco_sort.keras")


## 1. Chargement des datasets train/val

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names  # ordre alphabetique : bleue, d3e, jaune, marron, verte
print("Classes (ordre utilise par le modele) :", class_names)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(autotune)
val_ds = val_ds.prefetch(autotune)


## 2. Augmentation de donnees (anticipe l'ecart Kaggle / Jumia)

In [ ]:
def build_augmentation():
    return models.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.15),
            layers.RandomZoom(0.2),
            layers.RandomTranslation(0.1, 0.1),
            layers.RandomContrast(0.3),
            layers.RandomBrightness(0.2),
        ],
        name="data_augmentation",
    )

# Petit apercu de l'augmentation sur un batch
import matplotlib.pyplot as plt

augmentation_preview = build_augmentation()
sample_images, _ = next(iter(train_ds))

plt.figure(figsize=(10, 10))
for i in range(9):
    augmented = augmentation_preview(sample_images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Exemples d'augmentation sur une meme image")
plt.show()


## 3. Construction du modele (MobileNetV2 + tete de classification)

In [ ]:
def build_model(num_classes: int):
    augmentation = build_augmentation()

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False  # phase 1 : backbone gele

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augmentation(inputs)
    # Equivalent a mobilenet_v2.preprocess_input ([0,255] -> [-1,1]), mais via
    # une couche Keras native (evite l'echec de sauvegarde HDF5 du a l'appel
    # de fonction du module dans le graphe).
    x = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0)(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(num_classes=len(class_names))
model.summary()


## 4. Phase 1 — entrainement de la tete (backbone gele)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_HEAD = 10
history_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD)


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_head.history["accuracy"], label="train")
plt.plot(history_head.history["val_accuracy"], label="val")
plt.title("Accuracy - Phase 1")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_head.history["loss"], label="train")
plt.plot(history_head.history["val_loss"], label="val")
plt.title("Loss - Phase 1")
plt.legend()
plt.show()


## 5. Phase 2 — fine-tuning des dernieres couches du backbone

In [ ]:
FINETUNE_LAYERS = 30
EPOCHS_FINETUNE = 10

base_model.trainable = True
for layer in base_model.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # LR tres faible
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE)


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_finetune.history["accuracy"], label="train")
plt.plot(history_finetune.history["val_accuracy"], label="val")
plt.title("Accuracy - Phase 2 (fine-tuning)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_finetune.history["loss"], label="train")
plt.plot(history_finetune.history["val_loss"], label="val")
plt.title("Loss - Phase 2 (fine-tuning)")
plt.legend()
plt.show()


## 6. Evaluation finale sur le val split Kaggle

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Val Kaggle -> loss={loss:.4f} accuracy={acc:.4f}")


## 7. Sauvegarde du modele

**Ne lance cette cellule que si les resultats ci-dessus te conviennent.**
Une fois satisfait, previens-moi : je convertirai ce notebook en script
`src/ml/train.py` reproductible pour le livrable final.

In [ ]:
OUTPUT_MODEL.parent.mkdir(parents=True, exist_ok=True)
model.save(OUTPUT_MODEL)

class_names_path = OUTPUT_MODEL.parent / "class_names.json"
with open(class_names_path, "w") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("Modele sauvegarde :", OUTPUT_MODEL)
print("Ordre des classes sauvegarde :", class_names_path)
